# LeetCode 1451 — Rearrange Words in a Sentence bằng Python

Link đề: https://leetcode.com/problems/rearrange-words-in-a-sentence/ · Độ khó: **Medium** · Chủ đề: String, Sorting

Skeleton LeetCode:

```python
class Solution:
    def arrangeWords(self, text: str) -> str:
        ...
```

## 1. Mục tiêu học

1. Tách / ghép câu bằng `str.split` và `" ".join`, xử lý **chữ hoa – chữ thường** đúng chỗ (`lower`, `upper`, `capitalize`).
2. Hiểu **sắp xếp ổn định (stable sort)**: phần tử "bằng nhau" theo khoá giữ nguyên thứ tự ban đầu — và biết `sorted`/`list.sort` của Python (Timsort) là ổn định.
3. Dùng tham số `key=len` và kỹ thuật **decorate–sort** với khoá `(độ dài, chỉ số)` khi không chắc thuật toán có ổn định.
4. Viết **counting/bucket sort theo độ dài** đạt `O(n)`.
5. Tránh lỗi: quên hạ chữ hoa của từ đầu cũ, dùng `title()`, phá thứ tự gốc khi hoà, làm mất từ trùng.

## 2. Đề bài & ví dụ

Cho câu `text` gồm các từ cách nhau **đúng một** dấu cách; **chữ cái đầu tiên viết hoa**, còn lại là chữ thường.

Sắp xếp lại các từ theo **độ dài tăng dần**; nếu hai từ **cùng độ dài** thì giữ **thứ tự xuất hiện ban đầu**. Trả về câu mới theo cùng định dạng (chữ cái đầu viết hoa, phần còn lại chữ thường).

**Ví dụ 1:** `"Leetcode is cool"` → `"Is cool leetcode"`.

**Ví dụ 2:** `"Keep calm and code on"` → `"On and keep calm code"` (ba từ dài 4 giữ thứ tự `keep, calm, code`).

**Ví dụ 3:** `"To be or not to be"` → `"To be or to be not"`.

**Ràng buộc:** `text` bắt đầu bằng chữ hoa, sau đó là chữ thường và dấu cách đơn giữa các từ; `1 <= text.length <= 10^5`.

In [1]:
import random
import string
import time
from collections import defaultdict
from typing import List

examples = [
    ("Leetcode is cool", "Is cool leetcode"),
    ("Keep calm and code on", "On and keep calm code"),
    ("To be or not to be", "To be or to be not"),
]
for text, want in examples:
    print(f"{text!r:28} -> {want!r}")

'Leetcode is cool'           -> 'Is cool leetcode'
'Keep calm and code on'      -> 'On and keep calm code'
'To be or not to be'         -> 'To be or to be not'


## 3. Kiến thức nền

### 3.1 `split` và `join`

`text.split()` tách theo khoảng trắng thành list từ; `" ".join(words)` ghép lại với một dấu cách.

In [2]:
text = "Keep calm and code on"
words = text.split()
print(words, len(words))
print(" ".join(words) == text)

['Keep', 'calm', 'and', 'code', 'on'] 5
True


### 3.2 Chữ hoa – chữ thường: `lower`, `upper`, `capitalize`, `title`

- `s.lower()` / `s.upper()`: cả chuỗi.
- `s[0].upper() + s[1:]`: chỉ viết hoa ký tự đầu.
- `s.capitalize()`: viết hoa ký tự đầu **và hạ mọi ký tự còn lại** — với đề này tiện vì phần còn lại vốn là chữ thường.
- `s.title()`: viết hoa đầu **mỗi từ** — **không** phải điều đề muốn.

In [3]:
s = "is cool leetcode"
print(s[0].upper() + s[1:])
print(s.capitalize())
print(s.title(), "<- title() viết hoa mọi từ")
print("Leetcode".lower(), "Leetcode"[0].lower() + "Leetcode"[1:])

Is cool leetcode
Is cool leetcode
Is Cool Leetcode <- title() viết hoa mọi từ
leetcode leetcode


### 3.3 `sorted(..., key=...)`

`sorted(iterable, key=f)` sắp theo giá trị `f(x)`. `key=len` sắp theo độ dài. Trả về list mới; `list.sort(key=...)` sắp tại chỗ.

In [4]:
words = ["keep", "calm", "and", "code", "on"]
print(sorted(words, key=len))
print(sorted(words), "<- không key: theo bảng chữ cái")

['on', 'and', 'keep', 'calm', 'code']
['and', 'calm', 'code', 'keep', 'on'] <- không key: theo bảng chữ cái


### 3.4 Sắp xếp **ổn định** (stable)

Một thuật toán sắp xếp là **ổn định** nếu các phần tử có khoá bằng nhau giữ nguyên thứ tự tương đối như ban đầu. Python đảm bảo `sorted` và `list.sort` **luôn ổn định** (Timsort) — kể cả khi `reverse=True`. Đây chính là điều kiện "cùng độ dài thì giữ thứ tự gốc" của đề.

In [5]:
pairs = [("keep", 1), ("on", 2), ("calm", 3), ("and", 4), ("code", 5)]
print(sorted(pairs, key=lambda p: len(p[0])))
print(sorted(pairs, key=lambda p: len(p[0]), reverse=True), "<- reverse vẫn giữ thứ tự gốc khi hoà")

[('on', 2), ('and', 4), ('keep', 1), ('calm', 3), ('code', 5)]
[('keep', 1), ('calm', 3), ('code', 5), ('and', 4), ('on', 2)] <- reverse vẫn giữ thứ tự gốc khi hoà


### 3.5 `enumerate` và khoá tuple

Khoá có thể là tuple: Python so sánh tuple **từ trái sang phải**. Khoá `(len(w), i)` nghĩa là: theo độ dài, hoà thì theo chỉ số gốc — buộc thứ tự đúng kể cả với thuật toán sắp xếp không ổn định.

In [6]:
words = ["keep", "calm", "and", "code", "on"]
dec = [(len(w), i, w) for i, w in enumerate(words)]
print(dec)
print(sorted(dec))
print((4, 0) < (4, 3) < (5, 0))

[(4, 0, 'keep'), (4, 1, 'calm'), (3, 2, 'and'), (4, 3, 'code'), (2, 4, 'on')]
[(2, 4, 'on'), (3, 2, 'and'), (4, 0, 'keep'), (4, 1, 'calm'), (4, 3, 'code')]
True


## 4. Ý tưởng nền tảng / trực giác

Làm tay ví dụ 2 `"Keep calm and code on"`:

1. **Hạ chữ hoa** của từ đầu (vì nó có thể không còn đứng đầu): `keep calm and code on`.
2. Ghi độ dài và vị trí:

| vị trí | từ | độ dài |
|---|---|---|
| 0 | keep | 4 |
| 1 | calm | 4 |
| 2 | and | 3 |
| 3 | code | 4 |
| 4 | on | 2 |

3. Xếp theo độ dài, hoà thì theo vị trí: `on(2) and(3) keep(4,0) calm(4,1) code(4,3)`.
4. **Viết hoa** ký tự đầu của câu mới: `On and keep calm code`.

Ba bước "hạ – sắp ổn định – nâng" là toàn bộ bài toán.

In [7]:
text = "Keep calm and code on"
words = text.lower().split()
print("1. hạ chữ hoa:", words)
table = [(i, w, len(w)) for i, w in enumerate(words)]
print("2. (vị trí, từ, độ dài):", table)
ordered = [w for i, w, L in sorted(table, key=lambda t: (t[2], t[0]))]
print("3. sắp xếp:", ordered)
res = " ".join(ordered)
print("4. viết hoa:", res[0].upper() + res[1:])

1. hạ chữ hoa: ['keep', 'calm', 'and', 'code', 'on']
2. (vị trí, từ, độ dài): [(0, 'keep', 4), (1, 'calm', 4), (2, 'and', 3), (3, 'code', 4), (4, 'on', 2)]
3. sắp xếp: ['on', 'and', 'keep', 'calm', 'code']
4. viết hoa: On and keep calm code


## 5. Cách 1 — Brute Force / baseline: sắp xếp chèn (insertion sort) tự viết

**Ý tưởng:** tự cài đặt một thuật toán sắp xếp **ổn định** đơn giản: chèn từng từ vào phần đã sắp. Chỉ dịch qua những từ **dài hơn hẳn** (`>`), không vượt qua từ **bằng** độ dài ⇒ giữ thứ tự gốc.

In [8]:
def arrange_insertion(text: str) -> str:
    words = text.lower().split()
    for i in range(1, len(words)):
        cur = words[i]
        j = i - 1
        while j >= 0 and len(words[j]) > len(cur):   # dấu > (strict) giữ ổn định
            words[j + 1] = words[j]
            j -= 1
        words[j + 1] = cur
    res = " ".join(words)
    return res[0].upper() + res[1:]

for text, want in examples:
    print(repr(arrange_insertion(text)), arrange_insertion(text) == want)

'Is cool leetcode' True
'On and keep calm code' True
'To be or to be not' True


**Độ phức tạp:** Time `O(m²)` với `m` là số từ (xấu nhất khi từ dài đứng trước), cộng `O(n)` cho `lower/split/join` (n = độ dài chuỗi). Với `n = 10^5` có thể tới ~5·10⁴ từ ⇒ ~10⁹ bước: quá chậm.

**Nút thắt:** tự viết sắp xếp `O(m²)`. Python đã có sắp xếp ổn định `O(m log m)` viết bằng C.

In [9]:
rng = random.Random(0)
def random_text(m, rng, maxlen=8):
    ws = ["".join(rng.choice(string.ascii_lowercase) for _ in range(rng.randint(1, maxlen))) for _ in range(m)]
    s = " ".join(ws)
    return s[0].upper() + s[1:]

t3000 = random_text(3000, rng)
t = time.perf_counter()
arrange_insertion(t3000)
print(f"insertion sort, 3000 từ: {time.perf_counter()-t:.2f}s")

insertion sort, 3000 từ: 0.21s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — sort ổn định có sẵn:** điều kiện "cùng độ dài giữ thứ tự gốc" **chính là định nghĩa** của sắp xếp ổn định. Python đảm bảo `sorted(words, key=len)` ổn định ⇒ một dòng là đủ.

**Insight 2 — khoá phụ là chỉ số:** nếu không muốn dựa vào tính ổn định (hoặc ở ngôn ngữ khác), dùng khoá `(len(w), i)` — mọi khoá khác nhau nên thứ tự là duy nhất.

**Insight 3 — độ dài là số nguyên nhỏ:** độ dài từ ≤ n, nên có thể **xếp vào thùng theo độ dài** (bucket/counting sort) rồi đọc các thùng từ nhỏ đến lớn ⇒ `O(n)`. Đẩy từ vào thùng theo thứ tự xuất hiện ⇒ tự nhiên ổn định.

**Quy tắc định dạng:** **nếu** từ đầu đang viết hoa **thì** hạ nó trước khi sắp; **sau khi** ghép **thì** viết hoa ký tự đầu của kết quả.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — `sorted(key=len)` (dựa vào tính ổn định)

**Giải thích từng đoạn:**
1. `text.lower().split()` — hạ toàn bộ (đề đảm bảo chỉ ký tự đầu là hoa) và tách từ.
2. `sorted(words, key=len)` — ổn định nên hoà giữ thứ tự gốc.
3. Ghép và `capitalize()`.

In [10]:
def arrange_sorted(text: str) -> str:
    words = sorted(text.lower().split(), key=len)
    return " ".join(words).capitalize()

for text, want in examples:
    print(repr(arrange_sorted(text)), arrange_sorted(text) == want)

'Is cool leetcode' True
'On and keep calm code' True
'To be or to be not' True


**Code nộp LeetCode (cách 2)** — xem mục 9 (lời giải chính). **Độ phức tạp:** Time `O(n + m log m)`, Space `O(n)`.

### 7.2 Cách 3 — Decorate–sort với khoá `(độ dài, chỉ số)`

**Giải thích:** "trang trí" mỗi từ bằng tuple `(len, index, word)`, sắp tuple (so sánh từ trái sang phải), rồi "bóc" lấy từ. Thứ tự không phụ thuộc tính ổn định — tư duy cần thiết khi gợi ý của đề nói "store each word and their relative position".

In [11]:
def arrange_decorate(text: str) -> str:
    words = text.lower().split()
    decorated = sorted((len(w), i, w) for i, w in enumerate(words))
    res = " ".join(w for _, _, w in decorated)
    return res[0].upper() + res[1:]

class SolutionDecorate:
    def arrangeWords(self, text: str) -> str:
        words = text.lower().split()
        order = sorted(range(len(words)), key=lambda i: (len(words[i]), i))
        res = " ".join(words[i] for i in order)
        return res[0].upper() + res[1:]

for text, want in examples:
    print(arrange_decorate(text) == want, SolutionDecorate().arrangeWords(text) == want)

True True
True True
True True


`SolutionDecorate` sắp **chỉ số** thay vì tuple — tiết kiệm bộ nhớ một chút. **Độ phức tạp:** Time `O(n + m log m)`, Space `O(m)` thêm cho tuple/chỉ số.

### 7.3 Cách 4 — Bucket sort theo độ dài (`O(n)`)

**Giải thích từng đoạn:**
1. `buckets[L]` là list các từ dài `L`, thêm vào theo thứ tự xuất hiện (giữ ổn định).
2. Duyệt `L` từ 1 tới độ dài lớn nhất, nối các thùng.
3. Viết hoa ký tự đầu.

In [12]:
def arrange_bucket(text: str) -> str:
    words = text.lower().split()
    maxlen = max(len(w) for w in words)
    buckets = [[] for _ in range(maxlen + 1)]
    for w in words:
        buckets[len(w)].append(w)
    res = " ".join(w for b in buckets for w in b)
    return res[0].upper() + res[1:]

class SolutionBucket:
    def arrangeWords(self, text: str) -> str:
        buckets = defaultdict(list)
        for w in text.lower().split():
            buckets[len(w)].append(w)
        res = " ".join(w for L in sorted(buckets) for w in buckets[L])
        return res[0].upper() + res[1:]

for text, want in examples:
    print(arrange_bucket(text) == want, SolutionBucket().arrangeWords(text) == want)

True True
True True
True True


**Độ phức tạp:** `arrange_bucket`: Time `O(n)` (tổng độ dài từ + số thùng ≤ n), Space `O(n)`. `SolutionBucket` sắp các **độ dài khác nhau** — có tối đa `O(√n)` độ dài khác nhau (vì `1+2+…+k ≤ n`) nên gần như tuyến tính.

So sánh tốc độ với câu dài `10^5` ký tự (bỏ insertion sort):

In [13]:
rng = random.Random(1451)
big = random_text(20000, rng)[:10**5].rstrip()
print("độ dài:", len(big), "| số từ:", len(big.split()))
for name, f in [("sorted", arrange_sorted), ("decorate", arrange_decorate), ("bucket", arrange_bucket),
                ("SolutionBucket", SolutionBucket().arrangeWords)]:
    t = time.perf_counter()
    out = f(big)
    print(f"{name:15} {time.perf_counter()-t:.4f}s  {out[:40]}...")

độ dài: 100000 | số từ: 18183
sorted          0.0039s  M s d m c h o q b j w a s k j h s v x e ...
decorate        0.0079s  M s d m c h o q b j w a s k j h s v x e ...
bucket          0.0029s  M s d m c h o q b j w a s k j h s v x e ...
SolutionBucket  0.0020s  M s d m c h o q b j w a s k j h s v x e ...


## 8. TRACE từng bước

**Trace cách 1 (insertion sort)** trên `"Keep calm and code on"` — in mảng sau mỗi lần chèn:

| i | từ chèn | dịch qua | mảng sau bước |
|---|---|---|---|
| 1 | calm (4) | không (keep cũng 4, không dịch) | keep calm and code on |
| 2 | and (3) | calm, keep | and keep calm code on |
| 3 | code (4) | không | and keep calm code on |
| 4 | on (2) | code, calm, keep, and | on and keep calm code |

In [14]:
def trace_insertion(text):
    words = text.lower().split()
    print("ban đầu:", words)
    for i in range(1, len(words)):
        cur, j, moved = words[i], i - 1, []
        while j >= 0 and len(words[j]) > len(cur):
            moved.append(words[j])
            words[j + 1] = words[j]
            j -= 1
        words[j + 1] = cur
        print(f"i={i} chèn {cur!r} (len {len(cur)}) | dịch qua {moved} -> {words}")
    res = " ".join(words)
    return res[0].upper() + res[1:]

print(trace_insertion("Keep calm and code on"))

ban đầu: ['keep', 'calm', 'and', 'code', 'on']
i=1 chèn 'calm' (len 4) | dịch qua [] -> ['keep', 'calm', 'and', 'code', 'on']
i=2 chèn 'and' (len 3) | dịch qua ['calm', 'keep'] -> ['and', 'keep', 'calm', 'code', 'on']
i=3 chèn 'code' (len 4) | dịch qua [] -> ['and', 'keep', 'calm', 'code', 'on']
i=4 chèn 'on' (len 2) | dịch qua ['code', 'calm', 'keep', 'and'] -> ['on', 'and', 'keep', 'calm', 'code']
On and keep calm code


**Trace cách 4 (bucket)** trên `"To be or not to be"`:

In [15]:
def trace_bucket(text):
    words = text.lower().split()
    buckets = defaultdict(list)
    for i, w in enumerate(words):
        buckets[len(w)].append(w)
        print(f"từ {i}: {w!r} -> thùng {len(w)} | các thùng: {dict(buckets)}")
    order = [w for L in sorted(buckets) for w in buckets[L]]
    print("đọc thùng theo độ dài tăng:", order)
    res = " ".join(order)
    return res[0].upper() + res[1:]

print(trace_bucket("To be or not to be"))

từ 0: 'to' -> thùng 2 | các thùng: {2: ['to']}
từ 1: 'be' -> thùng 2 | các thùng: {2: ['to', 'be']}
từ 2: 'or' -> thùng 2 | các thùng: {2: ['to', 'be', 'or']}
từ 3: 'not' -> thùng 3 | các thùng: {2: ['to', 'be', 'or'], 3: ['not']}
từ 4: 'to' -> thùng 2 | các thùng: {2: ['to', 'be', 'or', 'to'], 3: ['not']}
từ 5: 'be' -> thùng 2 | các thùng: {2: ['to', 'be', 'or', 'to', 'be'], 3: ['not']}
đọc thùng theo độ dài tăng: ['to', 'be', 'or', 'to', 'be', 'not']
To be or to be not


## 9. Code nộp LeetCode

Lời giải chính (cách 2), đúng chữ ký `arrangeWords(self, text)`.

In [16]:
class Solution:
    def arrangeWords(self, text: str) -> str:
        words = sorted(text.lower().split(), key=len)   # sort ổn định
        return " ".join(words).capitalize()

for text, want in examples:
    assert Solution().arrangeWords(text) == want
print("Solution: 3 ví dụ đề OK")

Solution: 3 ví dụ đề OK


## 10. Vì sao thuật toán đúng

1. **Định dạng:** theo ràng buộc, chỉ ký tự đầu của `text` là chữ hoa. `lower()` đưa mọi từ về chữ thường — đúng dạng của mọi từ trừ từ đứng đầu kết quả. `capitalize()` (hoặc `res[0].upper() + res[1:]`) viết hoa đúng ký tự đầu và giữ phần còn lại chữ thường.
2. **Thứ tự:** kết quả cần là hoán vị các từ sao cho độ dài không giảm, và với hai từ cùng độ dài, từ xuất hiện trước đứng trước. Thứ tự này **duy nhất** (tương đương sắp theo khoá `(len, index)`, các khoá khác nhau từng đôi).
   - Cách 3 sắp đúng theo khoá đó.
   - Cách 2: sort ổn định theo `len` cho đúng thứ tự đó (định nghĩa của ổn định).
   - Cách 4: các thùng được đọc theo độ dài tăng; trong mỗi thùng từ được thêm theo thứ tự xuất hiện.
   - Cách 1: insertion sort chỉ dịch qua phần tử **dài hơn hẳn** nên không bao giờ đảo hai từ cùng độ dài ⇒ ổn định.
3. Không có từ nào bị mất hay nhân đôi vì mọi cách chỉ hoán vị list `words`.

## 11. Phân tích Time / Space

Gọi `n = len(text)`, `m` = số từ (`m ≤ (n+1)/2`).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Insertion sort | `O(n + m²)` | `O(n)` | quá chậm khi m ~ 5·10⁴ |
| 2. `sorted(key=len)` | `O(n + m log m)` | `O(n)` | ngắn, dựa vào stable |
| 3. Decorate `(len, i)` | `O(n + m log m)` | `O(n)` | không phụ thuộc stable |
| 4. Bucket theo độ dài | `O(n)` | `O(n)` | tuyến tính |

Mọi cách đều cần `O(n)` để tạo list từ và chuỗi kết quả.

## 12. Bộ test

Ví dụ đề, edge case (1 từ, 1 ký tự, mọi từ cùng độ dài, đã sắp sẵn, sắp ngược, nhiều từ trùng), so với brute force (insertion sort) trên dữ liệu ngẫu nhiên đúng định dạng đề (seed cố định), và kiểm tra **định dạng** đầu ra.

In [17]:
ALL = {
    "insertion": arrange_insertion,
    "sorted": arrange_sorted,
    "decorate": arrange_decorate,
    "bucket": arrange_bucket,
    "SolutionDecorate": SolutionDecorate().arrangeWords,
    "SolutionBucket": SolutionBucket().arrangeWords,
    "Solution": Solution().arrangeWords,
}
edge = [
    ("A", "A"),
    ("Hello", "Hello"),
    ("Abc def ghi", "Abc def ghi"),
    ("A bb ccc dddd", "A bb ccc dddd"),
    ("Dddd ccc bb a", "A bb ccc dddd"),
    ("Be be be b", "B be be be"),
    ("Leetcode is cool", "Is cool leetcode"),
]
def good_format(s):
    return s[0].isupper() and s[1:] == s[1:].lower() and "  " not in s and s == s.strip()

for text, want in examples + edge:
    for name, f in ALL.items():
        got = f(text)
        assert got == want, (name, text, got, want)
        assert good_format(got)
print("Ví dụ đề + edge case: OK, định dạng đúng")

Ví dụ đề + edge case: OK, định dạng đúng


In [18]:
rng = random.Random(2024)
for _ in range(1500):
    m = rng.randint(1, 25)
    text = random_text(m, rng, maxlen=rng.choice([1, 2, 4, 10]))
    want = arrange_insertion(text)
    for name, f in ALL.items():
        assert f(text) == want, (name, text)
    assert sorted(want.lower().split()) == sorted(text.lower().split())   # không mất / thêm từ
print("1500 câu ngẫu nhiên: mọi cách khớp insertion sort, là hoán vị các từ")

big_out = {f(big) for name, f in ALL.items() if name != "insertion"}
assert len(big_out) == 1
print("Câu 10^5 ký tự: các cách nhanh cho cùng kết quả")

1500 câu ngẫu nhiên: mọi cách khớp insertion sort, là hoán vị các từ


Câu 10^5 ký tự: các cách nhanh cho cùng kết quả


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên hạ chữ hoa của từ đầu cũ

Từ `"Leetcode"` chuyển xuống cuối vẫn giữ chữ `L` hoa.

In [19]:
def wrong_no_lower(text):
    words = sorted(text.split(), key=len)
    res = " ".join(words)
    return res[0].upper() + res[1:]

for text, want in examples[:2]:
    print(f"Sai : {wrong_no_lower(text)!r} | Đúng: {want!r}")

Sai : 'Is cool Leetcode' | Đúng: 'Is cool leetcode'
Sai : 'On and Keep calm code' | Đúng: 'On and keep calm code'


**Sửa:** `text.lower()` (hoặc hạ riêng ký tự đầu) **trước** khi sắp.

### Lỗi 2 — Hoà độ dài thì xếp theo bảng chữ cái

Dùng khoá `(len(w), w)` hoặc sắp theo bảng chữ cái trước — phá thứ tự gốc mà đề yêu cầu.

In [20]:
def wrong_alpha_tiebreak(text):
    words = sorted(text.lower().split(), key=lambda w: (len(w), w))   # SAI
    return " ".join(words).capitalize()

text = "Keep calm and code on"
print("Sai :", repr(wrong_alpha_tiebreak(text)))
print("Đúng:", repr(Solution().arrangeWords(text)))

Sai : 'On and calm code keep'
Đúng: 'On and keep calm code'


**Sửa:** khoá chỉ là `len` (dựa vào ổn định) hoặc `(len(w), index)`.

### Lỗi 3 — Dùng `title()` thay vì chỉ viết hoa ký tự đầu

`title()` viết hoa đầu **mọi** từ.

In [21]:
def wrong_title(text):
    return " ".join(sorted(text.lower().split(), key=len)).title()   # SAI

for text, want in examples:
    print(f"Sai : {wrong_title(text)!r} | Đúng: {want!r}")

Sai : 'Is Cool Leetcode' | Đúng: 'Is cool leetcode'
Sai : 'On And Keep Calm Code' | Đúng: 'On and keep calm code'
Sai : 'To Be Or To Be Not' | Đúng: 'To be or to be not'


**Sửa:** `capitalize()` hoặc `res[0].upper() + res[1:]`.

### Lỗi 4 — Dùng `set`/`dict` theo từ làm mất từ trùng

Muốn "lưu từ và độ dài" bằng dict `{word: len}` ⇒ từ lặp lại (`"to"`, `"be"`) bị gộp.

In [22]:
def wrong_dict(text):
    lens = {w: len(w) for w in text.lower().split()}   # SAI: khoá trùng bị ghi đè
    words = sorted(lens, key=lens.get)
    return " ".join(words).capitalize()

text = "To be or not to be"
print("Sai :", repr(wrong_dict(text)))
print("Đúng:", repr(Solution().arrangeWords(text)))

Sai : 'To be or not'
Đúng: 'To be or to be not'


**Sửa:** giữ nguyên **list** từ (có trùng); nếu cần lưu vị trí thì dùng `enumerate` (chỉ số là duy nhất).

### Lỗi 5 — Insertion sort dùng `>=` làm mất tính ổn định

Nếu khi chèn ta dịch qua cả từ **cùng độ dài** (`>=`), các từ cùng độ dài bị đảo thứ tự.

In [23]:
def wrong_unstable_insertion(text):
    words = text.lower().split()
    for i in range(1, len(words)):
        cur, j = words[i], i - 1
        while j >= 0 and len(words[j]) >= len(cur):   # SAI: >= phá ổn định
            words[j + 1] = words[j]
            j -= 1
        words[j + 1] = cur
    return " ".join(words).capitalize()

text = "Keep calm and code on"
print("Sai :", repr(wrong_unstable_insertion(text)), "| Đúng:", repr(Solution().arrangeWords(text)))

Sai : 'On and code calm keep' | Đúng: 'On and keep calm code'


**Sửa:** dùng `>` (strict).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề nói **"nếu bằng nhau thì giữ thứ tự ban đầu"** ⇒ **stable sort**; trong Python chỉ cần `sorted(key=...)`.
- Ở ngôn ngữ/thuật toán không đảm bảo ổn định ⇒ thêm **chỉ số gốc** vào khoá.
- Khoá là **số nguyên nhỏ** (độ dài, điểm số, tuổi) ⇒ cân nhắc **counting/bucket sort** `O(n)`.
- Bài xử lý chuỗi có định dạng: tách thành 3 bước **chuẩn hoá → xử lý → định dạng lại**.
- Nhớ `reverse=True` của Python vẫn ổn định (không đảo thứ tự các phần tử bằng nhau).

## 15. Biến thể của bài

1. **Độ dài giảm dần**, hoà giữ thứ tự gốc: `sorted(words, key=len, reverse=True)` (bài tập 4).
2. **Sort Characters by Frequency** (LeetCode 451): sắp ký tự theo tần suất — counting.
3. **Sorting the Sentence** (LeetCode 1859): mỗi từ kèm chỉ số cuối, khôi phục câu — bucket theo chỉ số.
4. **Khoá nhiều tầng:** theo số nguyên âm, rồi độ dài, rồi thứ tự gốc (bài tập 5).
5. **Custom Sort String** (LeetCode 791): sắp theo thứ tự cho trước — khoá là vị trí trong chuỗi `order`.

In [24]:
def sorting_the_sentence(s):
    # LeetCode 1859 — minh hoạ bucket theo chỉ số
    words = s.split()
    out = [None] * len(words)
    for w in words:
        out[int(w[-1]) - 1] = w[:-1]
    return " ".join(out)

print(sorting_the_sentence("is2 sentence4 This1 a3"))

This is a sentence


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Insertion sort | Tự viết sort ổn định | `O(n + m²)` | `O(n)` | Học ổn định, kiểm chứng |
| 2. `sorted(key=len)` | Dựa vào Timsort ổn định | `O(n + m log m)` | `O(n)` | Lời giải chuẩn trong Python |
| 3. Decorate `(len, i)` | Khoá phụ là chỉ số | `O(n + m log m)` | `O(n)` | Ngôn ngữ không có stable sort |
| 4. Bucket theo độ dài | Thùng theo độ dài | `O(n)` | `O(n)` | Tối ưu tuyệt đối |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output):** `arrangeWords("Hello my dear friend")` trả về gì?

**Bài 2 (tính tay):** với `"Keep calm and code on"`, viết danh sách tuple `(độ dài, chỉ số, từ)` **sau khi sắp xếp**.

**Bài 3 (điền TODO):** hoàn thiện `arrange_todo` theo cách bucket.

**Bài 4 (tự viết lại):** viết `arrange_desc(text)` sắp theo độ dài **giảm dần**, hoà giữ thứ tự gốc, cùng định dạng.

**Bài 5 (mini challenge):** viết `arrange_vowels(text)` sắp theo **số nguyên âm** (a, e, i, o, u) tăng dần; hoà thì theo **độ dài** tăng dần; hoà nữa thì giữ thứ tự gốc. Cùng định dạng đầu ra.

In [25]:
# Bài 3 — điền TODO (chỉ định nghĩa, không gọi)
def arrange_todo(text):
    words = text.lower().split()
    buckets = defaultdict(list)
    # TODO: thêm từng từ vào thùng theo độ dài (giữ thứ tự xuất hiện)
    # TODO: nối các thùng theo độ dài tăng dần thành chuỗi res
    res = ""
    # TODO: viết hoa ký tự đầu của res và trả về
    return res

## 18. Gợi ý 3 mức cho bài tập

**Bài 1:** (1) Hạ chữ: `hello my dear friend`. (2) Độ dài: 5, 2, 4, 6. (3) Sắp: `my dear hello friend` rồi viết hoa.

**Bài 2:** (1) Tạo `(len(w), i, w)` sau khi `lower()`. (2) Sắp tuple. (3) Thứ tự: `on, and, keep, calm, code`.

**Bài 3:** (1) `buckets[len(w)].append(w)`. (2) `" ".join(w for L in sorted(buckets) for w in buckets[L])`. (3) `res[0].upper() + res[1:]`.

**Bài 4:** (1) `reverse=True` vẫn ổn định. (2) Hoặc khoá `-len(w)`. (3) Đừng dùng `sorted(...)[::-1]` — đảo cả thứ tự các từ cùng độ dài.

**Bài 5:** (1) Hàm đếm nguyên âm `sum(c in "aeiou" for c in w)`. (2) Khoá tuple `(vowels(w), len(w))` + sort ổn định. (3) Kiểm chứng bằng insertion sort với cùng khoá.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
ans1 = Solution().arrangeWords("Hello my dear friend")
print("Bài 1:", ans1)
assert ans1 == "My dear hello friend"

# Bài 2
ws = "Keep calm and code on".lower().split()
ans2 = sorted((len(w), i, w) for i, w in enumerate(ws))
print("Bài 2:", ans2)
assert ans2 == [(2, 4, "on"), (3, 2, "and"), (4, 0, "keep"), (4, 1, "calm"), (4, 3, "code")]

# Bài 3
def arrange_todo_sol(text):
    words = text.lower().split()
    buckets = defaultdict(list)
    for w in words:
        buckets[len(w)].append(w)
    res = " ".join(w for L in sorted(buckets) for w in buckets[L])
    return res[0].upper() + res[1:]

for text, want in examples + edge:
    assert arrange_todo_sol(text) == want
print("Bài 3: OK")

Bài 1: My dear hello friend
Bài 2: [(2, 4, 'on'), (3, 2, 'and'), (4, 0, 'keep'), (4, 1, 'calm'), (4, 3, 'code')]
Bài 3: OK


In [27]:
# Bài 4
def arrange_desc(text):
    return " ".join(sorted(text.lower().split(), key=len, reverse=True)).capitalize()

def insertion_by_key(words, key):
    words = list(words)
    for i in range(1, len(words)):
        cur, j = words[i], i - 1
        while j >= 0 and key(words[j]) > key(cur):
            words[j + 1] = words[j]
            j -= 1
        words[j + 1] = cur
    return words

# Bài 5
def vowels(w):
    return sum(c in "aeiou" for c in w)

def arrange_vowels(text):
    words = sorted(text.lower().split(), key=lambda w: (vowels(w), len(w)))
    return " ".join(words).capitalize()

assert arrange_desc("Keep calm and code on") == "Keep calm code and on"
rng = random.Random(19)
for _ in range(600):
    text = random_text(rng.randint(1, 15), rng, maxlen=rng.choice([2, 5, 8]))
    ws = text.lower().split()
    assert arrange_desc(text) == " ".join(insertion_by_key(ws, lambda w: -len(w))).capitalize()
    assert arrange_vowels(text) == " ".join(insertion_by_key(ws, lambda w: (vowels(w), len(w)))).capitalize()
print("Bài 4:", arrange_desc("Keep calm and code on"))
print("Bài 5:", arrange_vowels("Keep calm and code on"), "| OK (600 test ngẫu nhiên)")

Bài 4: Keep calm code and on
Bài 5: On and calm keep code | OK (600 test ngẫu nhiên)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "hạ – sắp ổn định – nâng":**

```python
words = sorted(text.lower().split(), key=len)    # stable
return " ".join(words).capitalize()
```

**Template bucket `O(n)`:**

```python
buckets = defaultdict(list)
for w in text.lower().split():
    buckets[len(w)].append(w)
res = " ".join(w for L in sorted(buckets) for w in buckets[L])
return res[0].upper() + res[1:]
```

**Checklist:**
- [ ] Hạ chữ hoa của từ đầu cũ trước khi sắp.
- [ ] Hoà độ dài giữ **thứ tự gốc** (stable / khoá có chỉ số), không theo bảng chữ cái.
- [ ] Chỉ viết hoa **ký tự đầu** của câu (không `title()`).
- [ ] Không làm mất từ trùng (không dùng set/dict theo từ).
- [ ] Đã thử câu 1 từ, mọi từ cùng độ dài, câu sắp ngược.